In [ ]:
# Dataset and data root config

dataset = 'monk1'
data_root = './data'

In [ ]:
import numpy as np

def mean_euclidean_error(y_true, y_pred):
    # Calculate the difference between true and predicted values
    diff = y_true - y_pred
    
    # Reshapes row vectors to column vectors for consistent norm calculation
    if diff.ndim == 1:
        diff = diff.reshape(-1, 1)

    # Calculate mean Euclidean error
    return np.linalg.norm(diff, axis=1).mean()

In [ ]:
from sklearn.metrics import make_scorer

# Create a scorer for use in model evaluation
mee_scorer = make_scorer(mean_euclidean_error, greater_is_better=False)

In [ ]:
# === 2. IMPORTS AND UTILITIES ===
from torch.utils.data import DataLoader, TensorDataset
from models import SVCModel, SVRModel
from sklearn.metrics import *
from sklearn.multioutput import MultiOutputRegressor
# Assuming data_loader.py functions are available/imported
from utils.data_loader import get_monk1_data, get_ml_cup_data    
    
# --- Utility function to extract data from DataLoader to NumPy arrays ---
def extract_data_to_numpy(data_loader):
    """
    Converts data from a PyTorch DataLoader into a flattened NumPy array pair (X, y).
    Targets (y) are returned in their original dimensionality (e.g., [N, M] for multi-output).
    """
    X_list = []
    y_list = []
    for X, y in data_loader:
        # Flatten the input (e.g., 28x28 image -> 784 features)
        X_list.append(X.view(X.size(0), -1).numpy()) 
        # Convert labels to NumPy
        y_list.append(y.numpy())
    
    X_data = np.concatenate(X_list)
    y_data = np.concatenate(y_list)
    
    # We return y_data as is (2D array, e.g., [N, 1] or [N, M]).
    return X_data, y_data

In [ ]:
from sklearn.preprocessing import StandardScaler

# Use StandardScaler for feature scaling with 0 mean and unit variance
scaler = StandardScaler()

In [ ]:
# === 3. DATA LOADING AND PREPARATION ===

from utils.data_loader import get_monk1_data, get_ml_cup_data
import sys

dataset_name = dataset
BATCH_SIZE = 1024 # Batch size for DataLoader (not used in SVM but for consistency) 

print(f"Loading dataset: {dataset_name.upper()}...")
    
# Determine task type and load data (DataLoader objects are returned)
if dataset_name == 'monk1':
    train_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_monk1_data(BATCH_SIZE, data_root)
    is_regression_task = False
    metric_name = "Test Accuracy (%)"

elif dataset_name == 'mlc25':
    train_loader, test_loader = get_ml_cup_data(BATCH_SIZE, data_root, test_ratio = 0.25, mps = False,scaler = scaler)
    is_regression_task = True
    metric_name = "Test MEE"

else:
    # This block handles the error if the dataset is outside the specified choices (monk1, mlc25).
    print("Unsupported dataset for SVM.")
    sys.exit(1)


# --- 3.2 Data Preparation for Scikit-learn ---

# Convert DataLoaders (PyTorch) to NumPy arrays (Scikit-learn)
X_train, y_train = extract_data_to_numpy(train_loader)
X_test, y_test = extract_data_to_numpy(test_loader)

print(f"Data loaded: Training samples={X_train.shape[0]}, Test samples={X_test.shape[0]}")

In [ ]:
# Verify shapes

X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.svm import SVC, SVR
from scipy.stats import loguniform

params = [
    # 1. Linear Kernel: C is the ONLY parameter. Gamma is implicitly 'scale' or ignored.
    {
        'kernel': ['linear'],
        'C': loguniform(1e-3, 1e2),
    },
    
    # 2. RBF/Poly Kernels with Discrete Gamma: Tests the two known heuristics.
    {
        'kernel': ['rbf', 'poly'],
        'C': loguniform(1e-3, 1e2),
        'gamma': ['scale', 'auto'], # Discrete strings only
    },
    
    # 3. RBF/Poly Kernels with Continuous Gamma: Randomly samples from the continuous range.
    {
        'kernel': ['rbf', 'poly'],
        'C': loguniform(1e-3, 1e2),
        'gamma': loguniform(1e-3, 1e1), # Continuous distribution object only
    },
]

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) if not is_regression_task else KFold(n_splits=5, shuffle=True, random_state=42)  

In [ ]:
# HP Randomized Search Configuration

# Max iter and cache size set for efficiency
base_estimator = SVC(max_iter=-1, cache_size=1000) if not is_regression_task \
            else SVR(max_iter=-1, cache_size=1000)

hp_search = RandomizedSearchCV(
    estimator = base_estimator,
    param_distributions = params,
    n_iter = 20,                     # Number of random configurations to try
    scoring = 'accuracy' if not is_regression_task else mee_scorer,
    cv = kfold, 
    n_jobs = -1,
    verbose = 1)                          


In [ ]:
if is_regression_task:
    # Wrap in MultiOutputRegressor if the dataset is mlc25
    final_model = MultiOutputRegressor(hp_search)  
else:
    final_model = hp_search

In [ ]:
final_model.fit(X_train, y_train)

In [ ]:
if is_regression_task:
    # Print estimators for each output if mlc25
    print(final_model.estimators_)
else:
    print(final_model.best_params_)
    

In [ ]:
# === 4. MODEL INITIALIZATION (FINAL RECONSTRUCTION) ===

# --- 4.0 Helper Function to Create Wrapper from Estimator ---
def create_model_from_estimator(best_estimator):
    """
    Extracts parameters from the best_estimator and creates an instance
    of the following custom wrapper (SVCModel or SVRModel).
    """
    # 1. Get parameters from the fitted model
    params = best_estimator.get_params()
    
    # 2. Determine the type and instantiate the correct wrapper
    if isinstance(best_estimator, SVC):
        return SVCModel(**params), 'svc'
    elif isinstance(best_estimator, SVR):
        return SVRModel(**params), 'svr'
    else:
        raise TypeError(f"Type unsupported for reconstruction: {type(best_estimator)}")

In [ ]:
# --- 4.1 Reconstruction Logic ---

final_models_list = []  # List to hold final model wrappers
is_multi_output = False # Flag to indicate multi-output scenario

# MLC25 case: check if final_model has 'estimators_' attribute typical of MultiOutput wrapper
if hasattr(final_model, 'estimators_'):
    print(f"Detected Multi-Output System ({len(final_model.estimators_)} targets).")
    is_multi_output = True
    
    # Iterate over each RandomizedSearchCV contained in the MultiOutput
    for i, search_obj in enumerate(final_model.estimators_):
        # Extract the winner for this specific target
        best_est = search_obj.best_estimator_
        
        # Create wrapper
        wrapper, m_type = create_model_from_estimator(best_est)
        final_models_list.append(wrapper)
        
        print(f"Target {i}: Configured {m_type.upper()} with C={wrapper.model.C:.4f}, gamma={wrapper.model.gamma}")

# MONK1 case: Single Output
else:
    print("Detected Single-Output System.")
    # final_model is directly the RandomizedSearchCV
    best_est = final_model.best_estimator_
    
    wrapper, m_type = create_model_from_estimator(best_est)
    final_models_list.append(wrapper)
    
    print(f"Configured {m_type.upper()} with C={wrapper.model.C:.4f}, gamma={wrapper.model.gamma}")

In [ ]:
# --- 4.2 Final Fitting and Prediction ---

print("\n--- Starting Final Refitting of Models ---")

if is_multi_output:
    for i, wrapper in enumerate(final_models_list):
        print(f"Fitting Target {i}...")
        # Note: y_train[:, i] takes only the i-th column
        wrapper.model.fit(X_train, y_train[:, i]) 
        
else:
    print("Fitting Single Model...")
    final_models_list[0].model.fit(X_train, y_train.ravel())

print("Refitting done.")

In [ ]:
# === 6. PREDICTION & EVALUATION ===

# 6.1 Generation of Predictions (Handles List vs Single Model Case)
print("\n--- Generating Predictions ---")

if is_multi_output:
    # Multi-Output Case: Iterate over the list of models
    column_preds = []
    for wrapper in final_models_list:
        pred = wrapper.model.predict(X_test)
        column_preds.append(pred)
    
    # Combine columns: (N_samples, N_targets)
    y_test_pred = np.column_stack(column_preds)
    
    y_test_eval = y_test 
    
else:
    # Single Output Case: Directly use the single model
    y_test_pred = final_models_list[0].model.predict(X_test)
    
    # Flatten y_test for comparison
    y_test_eval = y_test.ravel()

# Ensure y_test is in the correct shape for comparison
print(f"Predictions generated. Shape: {y_test_pred.shape}")

In [ ]:
# 6.2 Calculating Metrics

print("\n--- Calculating Metrics ---")

if is_regression_task:
    # Calculate MEE for Regression
    final_metric = mean_euclidean_error(y_test_eval, y_test_pred)
else:
    # Calculate Accuracy for Classification
    final_metric = accuracy_score(y_test_eval, y_test_pred) * 100.0
    
# 6.3 Risultato Finale
print(f"Final Test {metric_name}: {final_metric}")

In [ ]:
# --- ADDITIONAL VISUALIZATIONS AND METRICS ---

# 1) Classification
if not is_regression_task:
    try:
        y_scores = final_models_list[0].model.decision_function(X_test)
    except AttributeError:
        print("Model does not support decision_function; skipping ROC and AUC calculations.")
        y_scores = y_test_pred

    # Confusion Matrix
    cm = confusion_matrix(y_test_eval, y_test_pred)
    ConfusionMatrixDisplay(cm).plot()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test_eval, y_scores)
    RocCurveDisplay(fpr = fpr, tpr = tpr).plot()

    # AUC Score
    auc_score = roc_auc_score(y_test_eval, y_scores) * 100.0
    print(f"AUC Score (%): {auc_score:.4f}")